# Crawl Hanzii, giữ CVDICT làm nguồn gốc

Notebook này tạo các file CSV trung gian trước khi import vào PostgreSQL. CVDICT là dữ liệu gốc; dữ liệu Hanzii chỉ được lưu như phần bổ sung.

Chạy thử một batch nhỏ trước. Chỉ chạy toàn bộ sau khi đã xác nhận quyền sử dụng dữ liệu Hanzii và kiểm tra parser.

In [20]:
from __future__ import annotations

import asyncio
import csv
import html
import json
import re
import time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import quote
from urllib.robotparser import RobotFileParser

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
CVDICT_PATH = PROJECT_ROOT / 'backend' / 'app' / 'data' / 'CVDICT.u8'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'hanzii_crawl'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Chạy thử trước. Đặt None sau khi parser đã được kiểm tra.
MAX_WORDS = 20
SAMPLE_WORDS = ['字幕', '水果', '医院', '工作', '打', '看', '开', '学习', '中文', '喜欢']
REQUEST_DELAY_SECONDS = 1.5
CHECKPOINT_EVERY = 10
MAX_ATTEMPTS = 3
HEADLESS = True
USER_AGENT = 'MandarinFlowDictionaryImporter/1.0 (+https://mandarinflow.online)'
HANZII_URL = 'https://hanzii.net/search/word/{word}?hl=vi'
HANZII_HOME = 'https://hanzii.net/'
ENRICHMENT_PATH = OUTPUT_DIR / 'hanzii_enrichment.csv'
ERROR_PATH = OUTPUT_DIR / 'crawl_errors.csv'


In [21]:
CVDICT_PATTERN = re.compile(r'^(?P<traditional>\S+)\s+(?P<simplified>\S+)\s+\[(?P<pinyin>[^\]]+)\]\s+/((?P<meaning>.+))/\s*$')

def clean_meaning(value: str) -> str:
    parts = []
    for item in value.split('/'):
        item = item.strip()
        if item and not item.startswith('LT:'):
            parts.append(item)
    return '; '.join(parts)

def read_cvdict(path: Path) -> list[dict[str, str]]:
    rows = []
    seen = set()
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        match = CVDICT_PATTERN.match(line)
        if not match:
            continue
        row = {
            'traditional': match.group('traditional'),
            'simplified': match.group('simplified'),
            'pinyin_numbered': match.group('pinyin'),
            'cvdict_meaning': clean_meaning(match.group('meaning')),
        }
        key = (row['simplified'], row['traditional'], row['pinyin_numbered'])
        if key not in seen:
            rows.append(row)
            seen.add(key)
    return rows

cvdict_rows = read_cvdict(CVDICT_PATH)
len(cvdict_rows), cvdict_rows[:3]

(122595,
 [{'traditional': '%',
   'simplified': '%',
   'pinyin_numbered': 'pa1',
   'cvdict_meaning': 'phần trăm (Đài Loan)'},
  {'traditional': '2019冠狀病毒病',
   'simplified': '2019冠状病毒病',
   'pinyin_numbered': 'er4 ling2 yi1 jiu3 guan1 zhuang4 bing4 du2 bing4',
   'cvdict_meaning': 'COVID-19, bệnh coronavirus được xác định năm 2019'},
  {'traditional': '21三體綜合症',
   'simplified': '21三体综合症',
   'pinyin_numbered': 'er4 shi2 yi1 san1 ti3 zong1 he2 zheng4',
   'cvdict_meaning': 'bệnh tam nhiễm sắc thể; hội chứng Down'}])

In [22]:
def write_csv(path: Path, rows: list[dict[str, str]], fieldnames: list[str]) -> None:
    with path.open('w', encoding='utf-8-sig', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames, extrasaction='ignore', lineterminator='\n')
        writer.writeheader()
        writer.writerows(rows)

base_fields = ['traditional', 'simplified', 'pinyin_numbered', 'cvdict_meaning']
write_csv(OUTPUT_DIR / 'cvdict_base.csv', cvdict_rows, base_fields)
print('Đã xuất', len(cvdict_rows), 'dòng:', OUTPUT_DIR / 'cvdict_base.csv')

Đã xuất 122595 dòng: /home/datnguyen/Documents/GIT_PROJECT/learning_chinese_through_vid/youtube-language-learning/data/hanzii_crawl/cvdict_base.csv


In [23]:
robots_parser = None

def can_fetch(url: str) -> bool:
    global robots_parser
    try:
        if robots_parser is None:
            robots_parser = RobotFileParser()
            robots_parser.set_url('https://hanzii.net/robots.txt')
            robots_parser.read()
        return robots_parser.can_fetch(USER_AGENT, url)
    except (HTTPError, URLError, TimeoutError) as exc:
        raise RuntimeError(f'Không đọc được robots.txt của Hanzii: {exc}') from exc

try:
    from playwright.async_api import TimeoutError as PlaywrightTimeoutError
    from playwright.async_api import async_playwright
except ImportError as exc:
    raise RuntimeError('Cài Playwright trước: %pip install playwright && playwright install chromium') from exc

def normalize_lines(text: str) -> list[str]:
    text = html.unescape(text).replace('\xa0', ' ')
    return [re.sub(r'\s+', ' ', line).strip() for line in text.splitlines() if line.strip()]

def is_pinyin_line(line: str) -> bool:
    line = normalize_pinyin(line)
    tokens = line.split()
    if not 1 <= len(tokens) <= 12:
        return False
    valid_token = re.compile(r"[A-Za-züÜāáǎàēéěèīíǐìōóǒòūúǔùǖǘǚǜv]+[1-5]?")
    tone_token = re.compile(r'[1-5āáǎàēéěèīíǐìōóǒòūúǔùǖǘǚǜ]')
    return all(valid_token.fullmatch(token) for token in tokens) and any(tone_token.search(token) for token in tokens)

def normalize_pinyin(line: str) -> str:
    bracket_match = re.search(r'\[\s*([A-Za-züÜāáǎàēéěèīíǐìōóǒòūúǔùǖǘǚǜv1-5 .-]+?)\s*\]', line)
    if bracket_match:
        return bracket_match.group(1).strip()
    return re.sub(r'^[\(]\s*|\s*[\)]$', '', line.strip())

SECTION_LABELS = {
    'Ví dụ', 'Ví dụ câu', 'Ngữ pháp', 'Kết hợp từ', 'Cụm từ', 'Từ ghép:', 'Từ ghép', 'Các từ gợi ý', 'Hán tự',
    'Hanzi', 'Example', 'Grammar', 'Collocations', 'Word', 'Information',
}
NAVIGATION_LINES = {
    'Trang chủ', 'Từ vựng', 'Hán tự', 'Ví dụ', 'Ngữ pháp', 'Kết hợp từ',
    'Đăng nhập', 'Đăng ký', 'Tìm kiếm', 'Di chuyển từ', 'Tạo sổ tay',
}

PARTS_OF_SPEECH = {
    'danh từ', 'động từ', 'tính từ', 'trạng từ', 'đại từ', 'giới từ',
    '名词', '动词', '形容词', '副词', '代词', '介词',
}

def contains_chinese(line: str) -> bool:
    return bool(re.search(r'[\u3400-\u9fff]', line))

def is_headword_line(line: str, word: str) -> bool:
    return line == word or line.startswith(f'{word}【') or line.startswith(f'{word} [')

def section_lines(lines: list[str], labels: set[str]) -> list[str]:
    matching_indexes = [i for i, line in enumerate(lines) if line in labels]
    start = matching_indexes[-1] + 1 if matching_indexes else -1
    if start < 0:
        return []
    result = []
    for line in lines[start:]:
        if line in SECTION_LABELS and line not in labels:
            break
        result.append(line)
    return result

def parse_triplets(lines: list[str], key_names: tuple[str, str, str]) -> list[dict[str, str]]:
    result = []
    index = 0
    while index < len(lines):
        if not contains_chinese(lines[index]):
            index += 1
            continue
        chinese = lines[index]
        pinyin_index = next((j for j in range(index + 1, min(index + 4, len(lines))) if is_pinyin_line(lines[j])), None)
        if pinyin_index is None or pinyin_index + 1 >= len(lines):
            index += 1
            continue
        meaning = lines[pinyin_index + 1]
        if meaning in SECTION_LABELS or contains_chinese(meaning):
            index = pinyin_index + 1
            continue
        result.append(dict(zip(key_names, [chinese, normalize_pinyin(lines[pinyin_index]), meaning])))
        index = pinyin_index + 2
    return result[:5]

def parse_hanzii_text(word: str, rendered_text: str) -> dict[str, str]:
    lines = normalize_lines(rendered_text)
    word_index = next((i for i, line in enumerate(lines) if is_headword_line(line, word)), -1)
    pinyin_index = next((i for i in range(word_index + 1, min(len(lines), word_index + 8)) if is_pinyin_line(lines[i])), -1)
    if word_index < 0 or pinyin_index < 0:
        return {'word': word, 'hanzii_pinyin': '', 'hanzii_meaning': '', 'hanzii_part_of_speech': '', 'hanzii_collocations_json': '[]', 'hanzii_examples_json': '[]', 'raw_text': '\n'.join(lines)[:12000], 'status': 'no_result'}
    pinyin = normalize_pinyin(lines[pinyin_index])
    start = pinyin_index + 1
    pos_index = next((i for i in range(start, len(lines)) if lines[i].lower() in PARTS_OF_SPEECH), -1)
    part_of_speech = lines[pos_index] if pos_index >= 0 else ''
    meaning_candidates = []
    noise = {'AI', '0', 'Đầy đủ', 'Mở khóa', 'Phát âm', 'Góp ý', 'Báo lỗi', 'Độ phổ biến', 'Hình ảnh', 'Tập viết'}
    if pos_index >= 0:
        for line in lines[pos_index + 1:]:
            if line in {'Mở khóa', 'Lượng từ', 'Từ ghép:', 'Từ ghép', 'Các từ gợi ý', 'Bính âm:'}:
                break
            if line in noise or line == word or is_pinyin_line(line) or contains_chinese(line):
                continue
            if 2 <= len(line) <= 240 and not line.startswith(('©', 'http://', 'https://')):
                meaning_candidates.append(line)
            if meaning_candidates:
                break
    meanings = meaning_candidates
    collocations = parse_triplets(section_lines(lines, {'Từ ghép:', 'Từ ghép', 'Các từ gợi ý', 'Cụm từ', 'Kết hợp từ', 'Collocations'}), ('text', 'pinyin', 'meaning'))
    examples = parse_triplets(section_lines(lines, {'Ví dụ', 'Ví dụ câu', 'Example'}), ('chinese', 'pinyin', 'vietnamese'))
    # Chỉ đánh dấu thành công khi có nội dung nghĩa thực tế, tránh lưu trang shell.
    meaning = '; '.join(meanings)
    return {
        'word': word,
        'hanzii_pinyin': pinyin,
        'hanzii_meaning': meaning,
        'hanzii_part_of_speech': part_of_speech,
        'hanzii_collocations_json': json.dumps(collocations, ensure_ascii=False),
        'hanzii_examples_json': json.dumps(examples, ensure_ascii=False),
        'raw_text': '\n'.join(lines)[:12000],
        'status': 'success' if meaning else 'needs_parser_review',
    }

async def crawl_word(page, word: str) -> dict[str, str]:
    url = HANZII_URL.format(word=quote(word))
    if not can_fetch(url):
        raise PermissionError(f'robots.txt không cho phép crawl: {url}')
    last_row = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            if page.url == 'about:blank' or 'hanzii.net' not in page.url:
                await page.goto(HANZII_HOME, wait_until='domcontentloaded', timeout=30000)
            search_box = page.locator('input:visible').first
            await search_box.wait_for(state='visible', timeout=15000)
            await search_box.fill(word)
            await search_box.press('Enter')
            result = page.get_by_text(word, exact=True).first
            try:
                await result.wait_for(state='visible', timeout=5000)
            except PlaywrightTimeoutError:
                result = page.get_by_text(re.compile(rf'^{re.escape(word)}(?:【[^】]+】)?$')).first
                await result.wait_for(state='visible', timeout=5000)
            rendered_text = await page.locator('body').inner_text(timeout=10000)
            last_row = parse_hanzii_text(word, rendered_text)
            if last_row['status'] == 'success':
                return last_row
        except PlaywrightTimeoutError:
            last_row = None
        if attempt < MAX_ATTEMPTS:
            await page.goto(HANZII_HOME, wait_until='domcontentloaded', timeout=30000)
    return last_row or {'word': word, 'hanzii_pinyin': '', 'hanzii_meaning': '', 'hanzii_part_of_speech': '', 'hanzii_collocations_json': '[]', 'hanzii_examples_json': '[]', 'raw_text': '', 'status': 'no_result'}


## TEST CRAWL HANZII

Chạy các cell phía trên trước, sau đó chạy cell crawl ngay bên dưới để test. Cell này chỉ crawl các từ trong `SAMPLE_WORDS`; cell ghép CSV nằm phía dưới và có thể chạy riêng sau khi kiểm tra kết quả.

In [24]:
enrichment_fields = ['word', 'hanzii_pinyin', 'hanzii_meaning', 'hanzii_part_of_speech', 'hanzii_collocations_json', 'hanzii_examples_json', 'raw_text', 'status']

def read_existing(path: Path, fields: list[str]) -> dict[str, dict[str, str]]:
    if not path.exists():
        return {}
    with path.open(encoding='utf-8-sig', newline='') as file:
        return {row['word']: {key: row.get(key, '') for key in fields} for row in csv.DictReader(file) if row.get('word')}

def persist_checkpoint(enrichment_by_word, error_by_word) -> None:
    write_csv(ENRICHMENT_PATH, list(enrichment_by_word.values()), enrichment_fields)
    write_csv(ERROR_PATH, list(error_by_word.values()), ['word', 'error_type', 'error'])

def is_valid_result(row: dict[str, str]) -> bool:
    meaning = row.get('hanzii_meaning', '')
    return (
        row.get('status') == 'success'
        and is_pinyin_line(row.get('hanzii_pinyin', ''))
        and bool(meaning.strip())
        and not any(token in meaning for token in ('Hình thái:', 'Góp ý', 'Mở khóa', 'Từ điển Trung Việt'))
    )

async def crawl_batch() -> tuple[dict[str, dict[str, str]], dict[str, dict[str, str]]]:
    enrichment_by_word = read_existing(ENRICHMENT_PATH, enrichment_fields)
    error_by_word = read_existing(ERROR_PATH, ['word', 'error_type', 'error'])
    words = list(dict.fromkeys(row['simplified'] for row in cvdict_rows))
    sample = (SAMPLE_WORDS if MAX_WORDS is not None else words)[:MAX_WORDS] if MAX_WORDS is not None else words
    pending = [word for word in sample if is_valid_result(enrichment_by_word.get(word, {}))]
    print(f'Tổng {len(sample)} từ; bỏ qua {len(pending)} từ đã crawl thành công')
    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(headless=HEADLESS)
        context = await browser.new_context(user_agent=USER_AGENT, locale='vi-VN')
        page = await context.new_page()
        try:
            processed = 0
            for index, word in enumerate(sample, start=1):
                if is_valid_result(enrichment_by_word.get(word, {})):
                    continue
                try:
                    row = await crawl_word(page, word)
                    enrichment_by_word[word] = row
                    error_by_word.pop(word, None)
                    print(f'[{index}/{len(sample)}] {row["status"]} {word}')
                except Exception as exc:
                    error_by_word[word] = {'word': word, 'error_type': type(exc).__name__, 'error': str(exc)}
                    print(f'[{index}/{len(sample)}] ERROR {word}: {exc}')
                processed += 1
                if processed % CHECKPOINT_EVERY == 0:
                    persist_checkpoint(enrichment_by_word, error_by_word)
                await asyncio.sleep(REQUEST_DELAY_SECONDS)
        finally:
            await context.close()
            await browser.close()
    persist_checkpoint(enrichment_by_word, error_by_word)
    return enrichment_by_word, error_by_word

enrichment_by_word, error_by_word = await crawl_batch()
print(f'Đã lưu {len(enrichment_by_word)} kết quả, lỗi {len(error_by_word)} dòng')

Tổng 10 từ; bỏ qua 9 từ đã crawl thành công
[10/10] success 喜欢
Đã lưu 30 kết quả, lỗi 0 dòng


In [14]:
# Ghép CVDICT và Hanzii; chỉ dùng các cột đã kiểm tra.
hanzii_by_word = enrichment_by_word
merged = []
for row in cvdict_rows:
    extra = hanzii_by_word.get(row['simplified'], {})
    merged.append({
        **row,
        'hanzii_pinyin': extra.get('hanzii_pinyin', ''),
        'hanzii_meaning': extra.get('hanzii_meaning', ''),
        'hanzii_part_of_speech': extra.get('hanzii_part_of_speech', ''),
        'hanzii_collocations_json': extra.get('hanzii_collocations_json', '[]'),
        'hanzii_examples_json': extra.get('hanzii_examples_json', '[]'),
        'hanzii_status': extra.get('status', 'not_crawled'),
    })

merged_fields = base_fields + ['hanzii_pinyin', 'hanzii_meaning', 'hanzii_part_of_speech', 'hanzii_collocations_json', 'hanzii_examples_json', 'hanzii_status']
write_csv(OUTPUT_DIR / 'dictionary_merged.csv', merged, merged_fields)
print('Đã xuất file hợp nhất:', OUTPUT_DIR / 'dictionary_merged.csv')

Đã xuất file hợp nhất: /home/datnguyen/Documents/GIT_PROJECT/learning_chinese_through_vid/youtube-language-learning/data/hanzii_crawl/dictionary_merged.csv


## Kiểm tra trước khi chạy toàn bộ

1. Cài Playwright và Chromium trong kernel của notebook:
   `%pip install -r notebooks/requirements.txt` rồi `!playwright install chromium`.
2. Chạy batch 20 từ và kiểm tra `hanzii_enrichment.csv`; các dòng hợp lệ phải có `status=success` và `hanzii_meaning` không rỗng.
3. Nếu parser đúng, đổi `MAX_WORDS = None` để crawl toàn bộ CVDICT. Notebook sẽ resume từ các dòng đã có `status=success`.
4. Chỉ import PostgreSQL sau khi kiểm tra CSV và xác nhận quyền sử dụng dữ liệu Hanzii.